# Módulo 44: Preparación de datos — Guía del ejercicio

En este ejercicio vas a importar y limpiar en **Power Query** un extracto de ventas real, con archivos CSV, TXT y Excel que traen los problemas típicos: encabezados ausentes, duplicados, filas basura, campos empaquetados y archivos repartidos en una carpeta.

- Si necesitas repasar los conceptos, consulta la lección [m44-preparacion-de-datos-leccion.md](m44-preparacion-de-datos-leccion.md).
- Cuando termines, compara tu trabajo con la solución: [m44-preparacion-de-datos-solucion.md](m44-preparacion-de-datos-solucion.md) (también en [PDF](m44-preparacion-de-datos-solucion.pdf)).

## Contexto

Una cadena de tiendas con presencia en diez países de América y Europa quiere un tablero de ventas en Power BI. El área de sistemas te entrega un extracto tal como sale de sus sistemas: las ventas de 2019 en un CSV, las de 2020 y 2021 en una carpeta con un archivo por año, el catálogo de países en un TXT y los catálogos de clientes, tiendas y productos en un libro de Excel.

Antes de dibujar un solo gráfico, necesitas que cada tabla tenga encabezados correctos, tipos de datos adecuados, claves sin duplicados y ninguna fila basura. Como el extracto se actualizará cada cierto tiempo, la limpieza debe quedar registrada como pasos de Power Query que se repitan solos al actualizar.

## Datos

Los archivos están en [`../practica-ventas/datos-ventas/`](../practica-ventas/datos-ventas/):

| Archivo | Contenido | Particularidades |
| --- | --- | --- |
| `ventas-2019.csv` | 300 000 ventas de 2019 | UTF-8 con BOM, separado por comas, con encabezado; fechas en formato `M/D/YYYY` |
| `ventas-por-anio/ventas-2020.csv` | 499 000 ventas de 2020 | Misma estructura que 2019 |
| `ventas-por-anio/ventas-2021.csv` | 700 000 ventas de 2021 | Misma estructura, pero cada línea termina en coma |
| `paises.txt` | Catálogo de países | Separado por `\|`, **sin encabezado** y con filas repetidas |
| `catalogo.xlsx` | Hojas `t_customer`, `t_store` y `t_product` | Ver la tabla siguiente |

Las tres tablas de ventas tienen estas columnas:

| Columna | Descripción |
| --- | --- |
| `ID_T` | Número de transacción (cada archivo empieza en 1) |
| `ID_Country`, `ID_Store`, `ID_Client`, `ID_Product` | Claves de país, tienda, cliente y producto |
| `Date` | Fecha de la venta |
| `Quantity` | Unidades vendidas |
| `Price` | Precio unitario |

Las hojas de `catalogo.xlsx` traen sus propios problemas:

| Hoja | Columnas | Particularidades |
| --- | --- | --- |
| `t_customer` | `ID_Client`, `first_name`, `last_name`, `zone` | `ID_Client` empaqueta cinco datos separados por `\|` (por ejemplo `ID\|658\|Hamlin/Zohrer\|mail:hzohrerih@over-blog.com\|+55-890-712-8982`); al final hay filas vacías y un pie de página; `zone` vale `?` en todas las filas |
| `t_store` | `ID_Store`, `Name_Store`, `Addres`, `Format`, `Status` | Dos filas antes del encabezado; `Format` escrito de varias formas (`aa`, `AAA`, `   A`); algunas tiendas aparecen dos veces, como `Open` y como `Closed` |
| `t_product` | `Product ID`, `Category`, `Sub-Category`, `Product Name`, `Active` | `Product ID` tiene la forma `ID-1000\|Product`; algunos productos aparecen dos veces, uno con `Active = 1` y otro con `Active = 0` |

## Instrucciones

- Trabaja en **Power BI Desktop**: crea un archivo `.pbix` nuevo y haz toda la limpieza en el Editor de Power Query (**Inicio → Transformar datos**). No modifiques los archivos de origen.
- Activa en **Vista** la calidad, la distribución y el perfil de columnas; te ayudarán a detectar duplicados y valores raros.
- Este notebook es el enunciado y tu bitácora: después de cada tarea, describe en la celda los pasos que seguiste y pega una captura de pantalla de la consulta resultante (en Jupyter puedes arrastrar la imagen a la celda markdown).
- El archivo [practica-ventas.pbix](../practica-ventas/practica-ventas.pbix) contiene el informe terminado de toda la sección. Ábrelo solo al final, para comparar; sus orígenes apuntan a rutas del autor, así que para actualizarlo tendrías que cambiarlas en **Transformar datos → Configuración de origen de datos**.

Entrega tu archivo `.pbix` y este notebook con tus respuestas.

## Tarea 1. Países

1. Importa `paises.txt` con el conector **Texto/CSV**, indicando `|` como delimitador.
2. Como el archivo no tiene encabezado, nombra las columnas `ID_Country` y `Country`, con tipos número entero y texto.
3. Elimina los duplicados para que quede una fila por país. Llama a la consulta `t_countries`.

*Describe aquí los pasos y añade una captura de pantalla.*

## Tarea 2. Clientes

1. Importa la hoja `t_customer` de `catalogo.xlsx` y promueve la primera fila a encabezado.
2. Divide `ID_Client` por el delimitador `|` y conserva el número de cliente como `ID_Client` (número entero), junto con el correo y el teléfono.
3. Quita las filas vacías y el pie de página del final, y las columnas que no aportan información (el prefijo `ID`, `zone`).
4. Comprueba que queda una fila por cliente.

*Describe aquí los pasos y añade una captura de pantalla.*

## Tarea 3. Productos

1. Importa la hoja `t_product` y promueve el encabezado.
2. Extrae de `Product ID` la clave numérica (el número entre `ID-` y `|`) y llámala `ID_Product`, como número entero.
3. Conserva solo los productos activos (`Active = 1`) y comprueba que `ID_Product` ya no tiene repetidos.

*Describe aquí los pasos y añade una captura de pantalla.*

## Tarea 4. Tiendas

1. Importa la hoja `t_store`, quita las filas que hay antes del encabezado y promueve la fila de encabezado.
2. Asigna los tipos: `ID_Store` número entero y el resto texto.
3. Deja una sola fila por tienda, quedándote con la versión `Open`.
4. Normaliza `Format` para que solo existan los valores `A`, `AA` y `AAA`.

*Describe aquí los pasos y añade una captura de pantalla.*

## Tarea 5. Ventas de 2019

1. Importa `ventas-2019.csv` con el conector **Texto/CSV** y promueve el encabezado.
2. Asigna los tipos: claves y `Quantity` como número entero, `Date` como fecha (con la configuración regional `en-US` si tu equipo está en español) y `Price` como moneda.
3. Quita `ID_T`, que no es una clave única cuando se juntan los tres archivos.

*Describe aquí los pasos y añade una captura de pantalla.*

## Tarea 6. Ventas de 2020 y 2021 desde una carpeta

1. Importa la carpeta `ventas-por-anio/` con el conector **Carpeta** y pulsa **Combinar y transformar datos**.
2. Asegúrate de leer solo las ocho columnas reales: la coma final de `ventas-2021.csv` no debe producir una columna vacía.
3. Aplica los mismos tipos que en la Tarea 5 y quita las columnas `Source.Name` e `ID_T`.

*Describe aquí los pasos y añade una captura de pantalla.*

## Tarea 7. Tabla de ventas única

1. **Anexa** las consultas de las Tareas 5 y 6 en una consulta nueva llamada `Sales`. Comprueba que tiene las ventas de 2019, 2020 y 2021.
2. Agrega una columna personalizada `Sales` con el importe de cada venta, $\text{Sales} = \text{Quantity} \times \text{Price}$, de tipo moneda.
3. Opcional: deshabilita la carga de las dos consultas intermedias para no guardar las mismas filas dos veces en el modelo.
4. Pulsa **Cerrar y aplicar**.

*Describe aquí los pasos y añade una captura de pantalla.*

## Antes de terminar

- [ ] `t_countries`, `t_customer`, `t_product` y `t_store` tienen una sola fila por clave y claves de tipo número entero.
- [ ] Las fechas de las ventas se interpretan bien (no hay errores en `Date`).
- [ ] `Sales` contiene las ventas de los tres años, sin columnas vacías, y la columna `Sales` es de tipo moneda.
- [ ] Cada tarea tiene su descripción y su captura.

Cuando termines, compara tu trabajo con la solución: [m44-preparacion-de-datos-solucion.md](m44-preparacion-de-datos-solucion.md).